In [0]:
%run ../env

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *
from pyspark.sql.types import *

import uuid
import copy
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from typing import List, Dict, Union
# import pysftp
from paramiko.ssh_exception import SSHException
import re

import paramiko
import io

from concurrent.futures import ThreadPoolExecutor, as_completed
import traceback
from queue import Queue


from Crypto.Cipher import AES
from Crypto.Util.Padding import pad
import os
import hashlib

import zipfile

import logging

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(threadName)s - %(levelname)s - %(message)s'
)

#downstream constant

In [0]:
config_database = get_env_config("config_database")
print(f"config_database: {config_database}")
downstream_backups_database = get_env_config("downstream_backups_database")
print(f"downstream_backups_database: {downstream_backups_database}")

In [0]:
# 从001开始. 格式为: 001, 002, 010, 099
DOWNSTREAM_KEY_SPLIT_FILE_INDEX  = "split_file_index"


DOWNSTREAM_CONDITION_ALL_MARKET = "ALL"
DOWNSTREAM_CONDITION_ALL_BRAND = "ALL"

DOWNSTREAM_DATA_MODE_TC_INCR = "TC_INCR"
DOWNSTREAM_DATA_MODE_FULL = "FULL"

DEFAULT_COMPRESSION = "zip"
DEFAULT_COMPRESSION_POSTFIX = ".zip"

CDC_OPERATION_TIME_FORMAT = "%Y-%m-%dT%H:%M:%S.%f%z"  #例: 2024-11-20T08:39:34.476+00:00

#common fun

In [0]:
def replace_func(options):
    def f(match):
        match_str = match.group(0)  # 获取完整的匹配字符串
        job_time = options.job_time
        # job_trigger_time = options.job_trigger_time
        
        if match_str == "{yyyy}":
            return job_time.strftime("%Y")
        elif match_str == "{HH}":
            return job_time.strftime("%H")
        elif match_str == "{MM}":
            return job_time.strftime("%m")
        elif match_str == "{dd}":
            return job_time.strftime("%d")
        elif match_str == "{yyyyMM}":
            return job_time.strftime("%Y%m")
        elif match_str == "{yyyyMMdd}":
            return job_time.strftime("%Y%m%d")
        elif match_str == "{yyyyMMddHHmmss}":
            return job_time.strftime("%Y%m%d%H%M%S")
        elif match_str == "{yyyy-MM-dd HH:mm:ss}":
            return job_time.strftime("%Y-%m-%d %H:%M:%S")
        elif match_str == "{HHmmss}":
            return job_time.strftime("%H%M%S")
        elif match_str == "{uuid}":
            return str(uuid.uuid4())
        # elif match_str == "{job_trigger_time}":
        #     return job_trigger_time.strftime("%Y%m%d%H%M%S%f")[:-3]
        else:
            return match_str  # 如果匹配不上，则不进行替换

    return f

def get_replace_callback_value(options, text):
    replace_callback = replace_func(options)
    result = re.sub(r"({yyyy}|{HH}|{MM}|{dd}|{yyyyMM}|{yyyyMMdd}|{yyyyMMddHHmmss}|{yyyy-MM-dd HH:mm:ss}|{HHmmss}|{uuid}|{job_trigger_time}|{cdc_operation_time})", replace_callback, text)
    # if result.find('{') >= 0:
    #     raise ValueError(f"{result} cannot be replaced")
    return result



def replace_secret(obj):
    if type(obj) != str:
        return obj

    kv_scope = get_env_config("kv_scope")

    keys = re.findall(r'[$][{](.*?)[}]', obj)
    for key in keys:
        secret = get_kv(key, kv_scope)
        obj = obj.replace("${" + key + "}", secret)
    return obj


def normalize_downstream_mode(mode):
    if mode is None:
        return None

    mode_str = str(mode).strip().upper()
    if mode_str == "":
        return None

    return mode_str


def get_table_latest_operation_time(table_name):
    """
    获取表最近一次版本时间, 用于 FULL 模式写入日志水位。
    如果表不支持 DESCRIBE HISTORY, 回退为当前时间，避免任务失败。
    """
    try:
        return spark.sql(f'''
                    SELECT max(timestamp)
                    FROM (DESCRIBE HISTORY {table_name})
                    '''
        ).collect()[0][0]
    except Exception as e:
        logging.warning(f"{table_name} 无法通过 DESCRIBE HISTORY 获取更新时间, 回退为当前时间. error: {e}")
        return datetime.now(tz=ZoneInfo("Asia/Shanghai"))


In [0]:
class DownstreamConfig:
    def __init__(self,
                 config_id: int,
                 downstream_type: str,
                 downstream_mode: str,
                 table_name: str,
                 table_business_keys: List[str],
                 marketcode: str,
                 brandcode: str,
                 condition_str: str,
                 out_file_folder: str,
                 out_file_name: str,
                 is_split_file: bool,
                 split_row_count: int,
                 value_map: Dict[str, str],
                 include_fields: List[str],
                 exclude_fields: List[str],
                 is_backups: bool,
                 is_active: bool,
                 is_zip: bool,
                 is_encrypt: bool,
                 encrypt_config: str,
                 kafka_topic_name: str,
                 comment_str: str,
                 create_time: datetime,
                 update_time: datetime
                 ):
        self.config_id = config_id
        self.downstream_type = downstream_type
        self.downstream_mode = downstream_mode
        self.table_name = table_name
        self.table_business_keys = table_business_keys
        self.marketcode = marketcode
        self.brandcode = brandcode
        self.condition_str = condition_str
        self.out_file_folder = out_file_folder
        self.out_file_name = out_file_name
        self.is_split_file = is_split_file
        self.split_row_count = split_row_count
        self.value_map = value_map
        self.include_fields = include_fields
        self.exclude_fields = exclude_fields
        self.is_backups = is_backups
        self.is_active = is_active
        self.is_zip = is_zip
        self.is_encrypt = is_encrypt
        self.encrypt_config = encrypt_config
        self.kafka_topic_name = kafka_topic_name
        self.comment_str = comment_str
        self.create_time = create_time
        self.update_time = update_time


    def __str__(self):
        return str(vars(self))

def get_downstreamConfig_list_from_DF(config_df):
    if config_df.isEmpty():
        return []
    else:
        config_list = [
            DownstreamConfig(
                config_id=row.config_id,
                downstream_type=row.downstream_type,
                downstream_mode=normalize_downstream_mode(getattr(row, "downstream_mode", None)),
                table_name=row.table_name,
                table_business_keys=row.table_business_keys,
                marketcode=row.marketcode,
                brandcode=row.brandcode,
                condition_str=row.condition_str,
                out_file_folder=row.out_file_folder,
                out_file_name=row.out_file_name,
                is_split_file=row.is_split_file,
                split_row_count=row.split_row_count,
                value_map=row.value_map,
                include_fields=row.include_fields,
                exclude_fields=row.exclude_fields,
                is_backups=row.is_backups,
                is_active=row.is_active,
                is_zip=row.is_zip,
                is_encrypt=row.is_encrypt,
                encrypt_config=row.encrypt_config,
                kafka_topic_name=row.kafka_topic_name,
                comment_str=row.comment_str,
                create_time=row.create_time,
                update_time=row.update_time
            )
            for row in config_df.collect()
        ]

        return config_list

    

In [0]:
def repartition_by_split_count(df, split_count):
    df = (df
        .withColumn("_row_number_id", row_number().over(Window.orderBy(lit("*"))))
        .withColumn("_split_partition_num", floor((col("_row_number_id") - 1)/split_count))
        .repartition("_split_partition_num")
        .drop("_row_number_id", "_split_partition_num")
    )

    return df


def save_to_downstream_log(downstream_config):
    schema = StructType([
        StructField("task_id", StringType(), True),
        StructField("config_id", LongType(), True),
        StructField("downstream_file_num", LongType(), True),
        StructField("downstream_row_count", LongType(), True),
        StructField("downstream_file_list", ArrayType(StringType()), True),
        StructField("first_cdc_operation_time", TimestampType(), True),
        StructField("second_cdc_operation_time", TimestampType(), True),
        StructField("comment_str", StringType(), True),
        StructField("create_time", TimestampType(), True),
        StructField("update_time", TimestampType(), True),
    ])

    current_time = datetime.now(tz=ZoneInfo("Asia/Shanghai"))

    log_data = {
        "task_id": downstream_config.task_id,
        "config_id": downstream_config.config_id,
        "downstream_file_num": getattr(downstream_config, 'downstream_file_num', None),
        "downstream_row_count": getattr(downstream_config, 'downstream_row_count', None),
        "downstream_file_list": getattr(downstream_config, 'outfile_path_list', None),
        "first_cdc_operation_time": downstream_config.first_cdc_operation_time,
        "second_cdc_operation_time": downstream_config.second_cdc_operation_time,
        "comment_str": "",
        "create_time": current_time,
        "update_time": current_time
    }

    log_df = spark.createDataFrame([log_data], schema)

    log_df.write.format('delta').mode('append').saveAsTable(f"{config_database}.downstream_log")

##encrypt

In [0]:
def encrypt_by_AES(original_io_data, encrypt_config_dict):
    '''
    encrypt_config_dict:
        {
         "encrypt_type": "AES",
         "secret_key": "${AK:AES_KEY}"
        }
    '''
    
    aes_key = hashlib.sha256(replace_secret(encrypt_config_dict["secret_key"]).encode('utf-8')).digest()
    aes_iv = os.urandom(16)

    # 1. 创建 AES 加密器对象，使用 CBC 模式
    cipher = AES.new(aes_key, AES.MODE_CBC, aes_iv)

    # 2. 读取 IO 数据并进行填充
    original_io_data.seek(0)
    io_data = original_io_data.read()
    padded_data = pad(io_data, AES.block_size)

    # 3. 对数据进行加密
    encrypted_data = cipher.encrypt(padded_data)

    # 4. 将加密后的数据存储到加密的字节流中
    encrypted_buffer = io.BytesIO()
    encrypted_buffer.write(aes_iv)
    encrypted_buffer.write(encrypted_data)

    # 5. 输出加密后的 IO 流（只为演示，实际应用中可以保存到文件等地方）
    encrypted_buffer.seek(0)

    return encrypted_buffer


encrypt_mapping = {
    "AES": encrypt_by_AES,
}



def encrypt_io_data(original_io_data, encrypt_config_str):
    '''
    encrypt_config_str: json格式字符串, 目前支持 AES 加密
        {"encrypt_type": "AES"
         "other_config": ""
        }
    '''
    encrypt_config_dict = json.loads(encrypt_config_str)
    encrypt_type = encrypt_config_dict["encrypt_type"]

    encrypt_fun = encrypt_mapping[encrypt_type]

    return encrypt_fun(original_io_data, encrypt_config_dict)

##compress

In [0]:
def compress_by_zip(original_io_data, archive_name):

    original_io_data.seek(0)

    # 将IO数据压缩成 ZIP 格式
    zip_buffer = io.BytesIO()
    with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zip_file:
        zip_file.writestr(archive_name, original_io_data.getvalue())

    zip_buffer.seek(0) 
    return zip_buffer

#sftp

In [0]:
# def upload_file_to_sftp(file_tuple_list, sftp_host, sftp_username, sftp_password):
#     '''
#     file_tuple_list:   [(local_file_path, sftp_file_path)]                                                         
#         local_file_path: 本地文件绝对路径
#         sftp_file_path: sftp文件绝对路径
#     '''

#     cnopts = pysftp.CnOpts()
#     cnopts.hostkeys = None

#     with pysftp.Connection(host=sftp_host, username=sftp_username, password=sftp_password, private_key=".ppk", cnopts=cnopts) as sftp:
#         for file_tuple in file_tuple_list:
#             local_file_path = file_tuple[0]
#             sftp_file_path = file_tuple[1]

#             try:
#                 sftp.put(local_file_path, sftp_file_path)
#             except SSHException as e:
#                 print("发生异常, 休眠一秒后进行重新发送, 异常信息: {}".format(e))
#                 time.sleep(1)   
#                 sftp.put(local_file_path, sftp_file_path)
#             except:
#                 raise



# def save_to_sftp(table_data, downstream_config):
    
#     # 1. 替换文件相关关键字
#     # 1.1  替换 out_file_folder
#     downstream_config.out_file_folder = get_replace_callback_value(downstream_config, downstream_config.out_file_folder)
#     # 1.2  替换 out_file_name
#     downstream_config.out_file_name = get_replace_callback_value(downstream_config, downstream_config.out_file_name)

#     if downstream_config.out_file_folder.endswith("/"):
#         sftp_file_path = downstream_config.out_file_folder + downstream_config.out_file_name
#     else:
#         sftp_file_path = downstream_config.out_file_folder + "/" +downstream_config.out_file_name


#     # 2. 写入本地系统
#     if downstream_config.is_split_file == True:
#         table_data = repartition_by_split_count(table_data, downstream_config.split_row_count)
#     else:
#         table_data = table_data.coalesce(1)

#     if BLOB_PATH.endswith("/"):
#         local_file_path_by_spark = BLOB_PATH + sftp_file_path
#     else:
#         local_file_path_by_spark = BLOB_PATH + "/" +sftp_file_path
#     local_file_path_by_spark = re.sub(f"{{{DOWNSTREAM_KEY_SPLIT_FILE_INDEX}}}", "", local_file_path_by_spark)

#     table_data.write.csv(local_file_path_by_spark, mode="overwrite", header = True)

#     # 3.生成 sftp split file 映射
#     local_file_path_list = [file_info.path for file_info in  dbutils.fs.ls(local_file_path_by_spark) if file_info.path.endswith(".csv")]
#     out_file_map_list = []  

#     for i in range(len(local_file_path_list)):
#         split_file_index = f"{(i+1):03}"
#         out_file_map_list.append( 
#             (local_file_path_list[i],  re.sub(f"{{{DOWNSTREAM_KEY_SPLIT_FILE_INDEX}}}", split_file_index, sftp_file_path)) 
#         )
    
#     downstream_config.out_file_map_list = out_file_map_list
#     downstream_config.out_file_path_list = [tup[1] for tup in out_file_map_list]

#     downstream_config.downstream_row_count = spark.read.csv(local_file_path_by_spark, header=True).count()
#     downstream_config.downstream_file_num = len(out_file_map_list)

#     # 4. 文件上传sftp
#     upload_file_to_sftp(out_file_map_list, sftp_host, sftp_username, sftp_password)

#     return True

In [0]:
def upload_file_to_sftp(sftp_file_path, io_data, sftp_host, sftp_username, sftp_password):
    '''
        sftp_file_path: sftp文件绝对路径
    '''

    transport = paramiko.Transport((sftp_host, 22))
    transport.connect(username=sftp_username, password=sftp_password)
    sftp = paramiko.SFTPClient.from_transport(transport)

    sftp.putfo(io_data, sftp_file_path)

    # # 关闭连接
    sftp.close()
    transport.close()



def save_to_sftp(table_data, downstream_config, task_inner_parallelism):
    '''
        table_data:  DataFrame. 需要下发的数据.
    '''
    
    # 1. 替换文件相关关键字
    # 1.1  替换 out_file_folder
    downstream_config.out_file_folder = get_replace_callback_value(downstream_config, downstream_config.out_file_folder)
    # 1.2  替换 out_file_name
    downstream_config.out_file_name = get_replace_callback_value(downstream_config, downstream_config.out_file_name)

    if downstream_config.out_file_folder.endswith("/"):
        sftp_file_path = downstream_config.out_file_folder + downstream_config.out_file_name
    else:
        sftp_file_path = downstream_config.out_file_folder + "/" +downstream_config.out_file_name


    # 2. 计算切分文件数
    pandas_df = table_data.toPandas()
    if downstream_config.is_split_file != True:
        num_split_file = 1
        downstream_config.split_row_count = len(pandas_df)
    else:
        num_split_file = (len(pandas_df) - 1) // (downstream_config.split_row_count) + 1


    # 3.上传文件
    out_file_path_list = Queue()

    def split_data_send_sftp(split_index):
        # 生成文件名
        split_file_index = f"{(split_index+1):03}"
        file_path = re.sub(f"{{{DOWNSTREAM_KEY_SPLIT_FILE_INDEX}}}", split_file_index, sftp_file_path)
        

        # 计算当前文件的起始和结束索引
        start = split_index * downstream_config.split_row_count
        end = (split_index + 1) * downstream_config.split_row_count
        logging.debug(f"start: {start}, end: {end}")

        # 文件上传sftp
        output = io.BytesIO()
        pandas_df.iloc[start:end].to_csv(output, index=False, header=True)
        output.seek(0)

        # 文件加密
        if downstream_config.is_encrypt == True:
            logging.info(f'文件加密.')
            output = encrypt_io_data(output, downstream_config.encrypt_config)

        # 文件压缩
        if downstream_config.is_zip == True:
            logging.info(f'文件压缩.')
            output = compress_by_zip(output, file_path.split("/")[-1])
            file_path = file_path + DEFAULT_COMPRESSION_POSTFIX

        output.seek(0)
        upload_file_to_sftp(file_path, output, sftp_host, sftp_username, sftp_password)
        out_file_path_list.put(file_path)

        # if downstream_config.is_zip == True:
        #     zip_file_path = file_path + DEFAULT_COMPRESSION_POSTFIX

        #     pandas_df.iloc[start:end].to_csv(output, index=False, header=True, 
        #                                      compression=dict(method=DEFAULT_COMPRESSION, archive_name= file_path.split("/")[-1]))
            
        #     output.seek(0)
        #     upload_file_to_sftp(zip_file_path, output, sftp_host, sftp_username, sftp_password)
        #     out_file_path_list.put(zip_file_path)

        # else:
        #     pandas_df.iloc[start:end].to_csv(output, index=False, header=True)
        #     output.seek(0)
        #     upload_file_to_sftp(file_path, output, sftp_host, sftp_username, sftp_password)
        #     out_file_path_list.put(file_path)
        
        return file_path

    # 3.1 并发上传文件
    if num_split_file  < task_inner_parallelism:
        task_inner_parallelism = num_split_file

    with ThreadPoolExecutor(max_workers=task_inner_parallelism) as task_inner_executor:
        future_to_success = {task_inner_executor.submit(split_data_send_sftp, split_index): split_index for split_index in range(num_split_file)}
        
        # 等待任务完成并获取结果
        for future in as_completed(future_to_success):
            split_index = future_to_success[future]
            try:
                file_path = future.result()
                logging.info(f'The {file_path} upload success.')
            except Exception as exc:
                logging.error(f'!!!!!!!!!! The upload fail.')
                raise exc

    downstream_config.out_file_path_list = list(out_file_path_list.queue)
    downstream_config.downstream_row_count = len(pandas_df)
    downstream_config.downstream_file_num = num_split_file


    return True

#local

In [0]:
def save_to_local(table_data, downstream_config, task_inner_parallelism):
    '''
        table_data:  DataFrame. 需要下发的数据.
    '''
    
    # 1. 替换文件相关关键字
    # 1.1  替换 out_file_folder
    downstream_config.out_file_folder = get_replace_callback_value(downstream_config, downstream_config.out_file_folder)
    # 1.2  替换 out_file_name
    downstream_config.out_file_name = get_replace_callback_value(downstream_config, downstream_config.out_file_name)

    if downstream_config.out_file_folder.endswith("/"):
        local_file_path = downstream_config.out_file_folder + downstream_config.out_file_name
    else:
        local_file_path = downstream_config.out_file_folder + "/" +downstream_config.out_file_name


    # 2. 计算切分文件数
    pandas_df = table_data.toPandas()
    if downstream_config.is_split_file != True:
        num_split_file = 1
        downstream_config.split_row_count = len(pandas_df)
    else:
        num_split_file = (len(pandas_df) - 1) // (downstream_config.split_row_count) + 1


    # 3.上传文件
    out_file_path_list = Queue()

    def split_data_send_local(split_index):
        # 生成文件名
        split_file_index = f"{(split_index+1):03}"
        file_path = re.sub(f"{{{DOWNSTREAM_KEY_SPLIT_FILE_INDEX}}}", split_file_index, local_file_path)
        

        # 计算当前文件的起始和结束索引
        start = split_index * downstream_config.split_row_count
        end = (split_index + 1) * downstream_config.split_row_count
        logging.debug(f"start: {start}, end: {end}")

        # 文件上传
        output = io.BytesIO()
        pandas_df.iloc[start:end].to_csv(output, index=False, header=True)
        output.seek(0)

        # 文件加密
        if downstream_config.is_encrypt == True:
            logging.info(f'文件加密.')
            output = encrypt_io_data(output, downstream_config.encrypt_config)

        # 文件压缩
        if downstream_config.is_zip == True:
            logging.info(f'文件压缩.')
            output = compress_by_zip(output, file_path.split("/")[-1])
            file_path = file_path + DEFAULT_COMPRESSION_POSTFIX

        output.seek(0)
        # upload_file_to_sftp(file_path, output, sftp_host, sftp_username, sftp_password)
        with open(file_path, 'wb') as file:
            file.write(output.getvalue())

        out_file_path_list.put(file_path)

        # 文件上传
        # if downstream_config.is_zip == True:
        #     zip_file_path = file_path + DEFAULT_COMPRESSION_POSTFIX
        #     pandas_df.iloc[start:end].to_csv(zip_file_path, index=False, header=True, 
        #                                      compression=dict(method=DEFAULT_COMPRESSION, archive_name= file_path.split("/")[-1]))
        #     out_file_path_list.put(zip_file_path)
        # else:
        #     pandas_df.iloc[start:end].to_csv(file_path, index=False, header=True)
        #     out_file_path_list.put(file_path)
        
        
        return file_path

    # 3.1 并发上传文件
    if num_split_file  < task_inner_parallelism:
        task_inner_parallelism = num_split_file

    with ThreadPoolExecutor(max_workers=task_inner_parallelism) as task_inner_executor:
        future_to_success = {task_inner_executor.submit(split_data_send_local, split_index): split_index for split_index in range(num_split_file)}
        
        # 等待任务完成并获取结果
        for future in as_completed(future_to_success):
            split_index = future_to_success[future]
            try:
                file_path = future.result()
                logging.info(f'The {file_path} upload success.')
            except Exception as exc:
                logging.error(f'!!!!!!!!!! The upload fail.')
                raise exc

    downstream_config.out_file_path_list = list(out_file_path_list.queue)
    downstream_config.downstream_row_count = len(pandas_df)
    downstream_config.downstream_file_num = num_split_file


    return True

#kafka

In [0]:
def save_to_kafka(table_data, downstream_config, task_inner_parallelism):
    '''
        table_data:  DataFrame. 需要下发的数据.
    '''
    kafka_brokers = get_env_config("target_kafka.kafka_brokers")
    topic_name = downstream_config.kafka_topic_name

    print(f"kafka_brokers: {kafka_brokers}, topic_name: {topic_name}")
    table_data.cache()
  
    try:

        # 写入 Kafka
        (table_data
        .write
        .format("kafka")
        .option("kafka.bootstrap.servers", kafka_brokers)
        .option("topic", topic_name) 
        .save()
        )
        logging.info(f"send to {topic_name} of Kafka({kafka_brokers}) success.")
        
    except Exception as e:
        logging.error(f'!!!!!!!!!! send to {topic_name} of Kafka({kafka_brokers}).')
        raise e
    
    downstream_config.downstream_row_count = table_data.count()
    table_data.unpersist()

    return True

#main fun

In [0]:
save_mapping = {
    "sftp": save_to_sftp,
    "local": save_to_local,
    "kafka": save_to_kafka
}

In [0]:
def save_backups(backups_data, downstream_config: 'DownstreamConfig'):

    marketcode = downstream_config.marketcode.lower()
    Process_time_str = downstream_config.job_time.strftime("%Y-%m-%d %H:%M:%S")

    backups_table = f"{downstream_backups_database}.{downstream_config.table_name.split('.')[-1]}_{marketcode}"
    
    (backups_data
        .withColumn("_TaskId", lit(downstream_config.task_id))
        .withColumn("_ProcessTime", lit(Process_time_str))
        .write.format('delta')
        .mode('append')
        .option("mergeSchema", "true")
        .saveAsTable(backups_table)
        )

    logging.info(f"{downstream_config.table_name} backups success.")


In [0]:
def get_ttable_name(ctable_name):
    table_parts = ctable_name.split(".")

    ct_name = table_parts.pop()

    if ct_name.lower().startswith('c_'):
        tt_name = "t_" + ct_name[2:]
    else:
        raise ValueError(f"{ctable_name} is not c table.")

    table_parts.append(tt_name)

    return ".".join(table_parts)


def get_ttable_data_by_ctable_name(ctable_name):
    ttable_name = get_ttable_name(ctable_name)

    version_data = spark.sql(f'''
                SELECT max(version), max(timestamp)
                FROM (DESCRIBE HISTORY {ttable_name}) 
                '''
    ).collect()[0]

    last_version = version_data[0]
    last_timestamp = version_data[1]

    ttable_df = (spark.table(f"{ttable_name} @v{last_version}")
        .withColumn("CDC_CODE", lit("I"))
        .withColumn("CDC_OperationTime", lit(last_timestamp))
    )

    return ttable_df
    

In [0]:
# 方案一: 多版本数据cdc code 按照根据不同情况处理
# cdc_code_map = {
#     "I_I":	"I",
#     "I_U":	"I",
#     "I_D":	"IGNORE",
#     "U_I":	"U",
#     "U_U":	"U",
#     "U_D":	"D",
#     "D_I":	"U",
#     "D_U":	"U",
#     "D_D":	"D"
# }

# @udf(returnType=StringType())
# def generate_cdc_code(first_cdc_code, second_cdc_code):
#     '''
#         cdc_code type: I,  U,  D. 
#         first_cdc_code	second_cdc_code	generate_cdc_code
#         I	            I	            I
#         I	            U	            I
#         I	            D	            IGNORE
#         U	            I	            U
#         U	            U	            U
#         U	            D	            D
#         D	            I	            U
#         D	            U	            U
#         D	            D	            D
#     '''

#     return cdc_code_map.get(f"{first_cdc_code}_{second_cdc_code}", None)
    

# def get_cdc_code(multi_version_df, table_business_keys):
#     cdc_operation_time_rank_window = Window.partitionBy(*table_business_keys).orderBy(col("CDC_OperationTime").desc())

#     return multi_version_df \
#         .withColumn("_cdc_operation_time_rank", row_number().over(cdc_operation_time_rank_window)) \
#         .withColumn("_min_rank", min(col("_cdc_operation_time_rank")).over(Window.partitionBy(*table_business_keys))) \
#         .withColumn("_max_rank", max(col("_cdc_operation_time_rank")).over(Window.partitionBy(*table_business_keys))) \
#         .groupBy(*table_business_keys) \
#         .agg(
#             max(when(col("_cdc_operation_time_rank") == col("_max_rank"), col("CDC_CODE"))).alias("first_cdc_code"),
#             max(when(col("_cdc_operation_time_rank") == col("_min_rank"), col("CDC_CODE"))).alias("second_cdc_code")
#         ) \
#         .withColumn("_generate_CDC_CODE", generate_cdc_code(col("first_cdc_code"), col("second_cdc_code")))



# def generate_incr_data_by_multi_version(multi_version_df, table_business_keys):
#     original_columns =  multi_version_df.columns
#     cdc_operation_time_rank_window = Window.partitionBy(*table_business_keys).orderBy(col("CDC_OperationTime").desc())

#     cdc_code_df = get_cdc_code(multi_version_df, table_business_keys)

#     return multi_version_df \
#         .withColumn("_cdc_operation_time_rank", row_number().over(cdc_operation_time_rank_window)) \
#         .filter(col("_cdc_operation_time_rank") == 1) \
#         .join(cdc_code_df, table_business_keys) \
#         .withColumn("CDC_CODE", col("_generate_CDC_CODE")) \
#         .drop("_cdc_operation_time_rank", "first_cdc_code", "second_cdc_code", "_generate_CDC_CODE") \
#         .filter(col("CDC_CODE") != "IGNORE") \
#         .select(*original_columns)



# 方案二: 多版本数据只根据主键取最新数据
def generate_incr_data_by_multi_version(multi_version_df, table_business_keys):
    if table_business_keys == None or len(table_business_keys) == 0:
        logging.info("主键为空, 发送所有版本数据".center(50,"="))
        return multi_version_df
    
    cdc_operation_time_rank_window = Window.partitionBy(*table_business_keys).orderBy(col("CDC_OperationTime").desc())

    return multi_version_df \
        .withColumn("_cdc_operation_time_rank", row_number().over(cdc_operation_time_rank_window)) \
        .filter(col("_cdc_operation_time_rank") == 1) \
        .drop("_cdc_operation_time_rank")


In [0]:
def send_data_to_downstream(config_obj, task_inner_parallelism):

    downstream_config =  copy.deepcopy(config_obj)
    # downstream_config.task_id = str(uuid.uuid4())
    downstream_config.job_time = datetime.now(tz=ZoneInfo("Asia/Shanghai"))
    downstream_config.downstream_mode = normalize_downstream_mode(getattr(downstream_config, "downstream_mode", None))

    # 1. 获取待下发数据
    if downstream_config.downstream_mode == DOWNSTREAM_DATA_MODE_FULL:
        logging.info("本次任务通过 FULL 模式下发全量数据".center(100,"="))
        table_data = spark.table(f"{downstream_config.table_name}")

        second_cdc_operation_time = get_table_latest_operation_time(downstream_config.table_name)

    elif downstream_config.downstream_mode == DOWNSTREAM_DATA_MODE_TC_INCR:
        # 1.1 判断通过 指定版本 还是 查询日志表 的方式计算增量数据
        # 1.1.1 指定版本
        if downstream_config.first_cdc_operation_time != None:
            logging.info("本次任务通过 指定版本 计算增量数据".center(100,"="))

            table_data = spark.table(f"{downstream_config.table_name}") \
                .filter(col("CDC_OperationTime") > downstream_config.first_cdc_operation_time)

            if downstream_config.second_cdc_operation_time != None:
                table_data = table_data.filter(col("CDC_OperationTime") <= downstream_config.second_cdc_operation_time)

        else:
            # 1.1.2 查询日志表
            logging.info("本次任务通过 查询日志表 计算增量数据".center(100,"="))

            # 1.1.2.1 读取日志获取 cdc_operation_time
            first_cdc_operation_time = (spark.table(f"{config_database}.downstream_log")
                .filter(col("config_id") == downstream_config.config_id)
                .agg(max(col("second_cdc_operation_time")))
                .collect()[0][0]
            )

            # 1.1.2.2 筛选数据
            # 非首次任务, 根据 cdc_operation_time 筛选数据
            if first_cdc_operation_time != None:
                logging.info(" 非首次任务执行 ".center(100,"="))
                downstream_config.first_cdc_operation_time = first_cdc_operation_time
                table_data = spark.table(f"{downstream_config.table_name}")
                table_data = table_data.filter(col("CDC_OperationTime") > first_cdc_operation_time)
            else:
                # 首次任务从对应t表获取最新数据
                logging.info(" 首次任务执行 ".center(100,"="))
                table_data = get_ttable_data_by_ctable_name(downstream_config.table_name)

        # 2. 跨版本数据合并
        if table_data.select(col("CDC_OperationTime")).distinct().count() > 1:
            logging.info("进行跨版本合并数据".center(50,"="))
            table_data = generate_incr_data_by_multi_version(table_data, downstream_config.table_business_keys)

        second_cdc_operation_time = table_data.agg(max(col("CDC_OperationTime"))).collect()[0][0]

    else:
        raise ValueError(f"downstream_mode invalid: {downstream_config.downstream_mode}. allowed: {DOWNSTREAM_DATA_MODE_TC_INCR}, {DOWNSTREAM_DATA_MODE_FULL}")

    downstream_config.second_cdc_operation_time = second_cdc_operation_time


    # 4. 字段映射
    # 4.1  执行 value_map
    if downstream_config.value_map != None: 
        for new_field_name, field_expr in downstream_config.value_map.items():
            table_data = table_data.withColumn(new_field_name, expr(field_expr))


    # 3.  执行 condition
    market_condition_segment = "1=1"
    if downstream_config.marketcode.upper() != DOWNSTREAM_CONDITION_ALL_MARKET:
        market_condition_segment = f" MarketCode = '{downstream_config.marketcode}' "
    
    brand_condition_segment = "1=1"
    if downstream_config.brandcode.upper() != DOWNSTREAM_CONDITION_ALL_BRAND:
        brand_condition_segment = f" BrandCode = '{downstream_config.brandcode}' "

    custom_condition_segment = "1=1"
    if downstream_config.condition_str != None and downstream_config.condition_str != "":
        custom_condition_segment = downstream_config.condition_str

    table_data = table_data.where(" and ".join([market_condition_segment, brand_condition_segment, custom_condition_segment]))


    if table_data.isEmpty():
        logging.info(downstream_config)
        logging.info("没有需要上传数据")
        return False



    # 4.2  执行 include_fields
    if downstream_config.include_fields != None and len(downstream_config.include_fields) >0:
        table_data = table_data.select(downstream_config.include_fields)

    elif downstream_config.exclude_fields != None and len(downstream_config.exclude_fields) >0:
        table_data = table_data.drop(*downstream_config.exclude_fields)
    

    # 5. 发送
    save_mapping[downstream_config.downstream_type](table_data, downstream_config, task_inner_parallelism)

    # 7. 写入日志 
    save_to_downstream_log(downstream_config)

    # 8.备份至备份表中
    if  downstream_config.is_backups == True: 
        logging.info("启用备份任务")
        save_backups(table_data, downstream_config)

    logging.info(downstream_config)

    return True